# IPL CASE STUDY - Data Analysis

## Prerequisite

#### Install numpy

In [1]:
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Usecases

### Task 0: Data Loading and Preprocessing

#### (1) Import necessary packages

In [2]:
import numpy as np

#### (2) Load the datasets using numpy

In [ ]:
deliveries_data = ipl_data = np.genfromtxt(
    '../dataset/deliveries.csv',
    delimiter=',',
    dtype=None,
    names=True,
    encoding='utf-8',
    invalid_raise=False
)

print("[Deliveries]: ",deliveries_data.shape)


[Deliveries]:  (260920,)


#### (3) Extract columns

In [4]:
match_ids = deliveries_data['match_id']
batting_teams = deliveries_data['batting_team']
batters = deliveries_data['batter']
bowlers = deliveries_data['bowler']
batsman_runs = deliveries_data['batsman_runs']
overs = deliveries_data['over']


print("[Match IDs]: ", match_ids)
print("[Batting Teams]: ", batting_teams)
print("[Batters]: ", batters)
print("[Bowlers]: ", bowlers)
print("[Batsman Runs]: ", batsman_runs)
print("[Overs]: ", overs)

[Match IDs]:  [ 335982  335982  335982 ... 1426312 1426312 1426312]
[Batting Teams]:  ['Kolkata Knight Riders' 'Kolkata Knight Riders' 'Kolkata Knight Riders'
 ... 'Kolkata Knight Riders' 'Kolkata Knight Riders'
 'Kolkata Knight Riders']
[Batters]:  ['SC Ganguly' 'BB McCullum' 'BB McCullum' ... 'VR Iyer' 'SS Iyer'
 'VR Iyer']
[Bowlers]:  ['P Kumar' 'P Kumar' 'P Kumar' ... 'Shahbaz Ahmed' 'Shahbaz Ahmed'
 'Shahbaz Ahmed']
[Batsman Runs]:  [0 0 0 ... 1 1 1]
[Overs]:  [ 0  0  0 ... 10 10 10]


### Task 1: Total Runs per Match

#### (1) Get unique match IDs

In [5]:
unique_matches = np.unique(match_ids)
print("[Unique Match IDs]: ", unique_matches)

[Unique Match IDs]:  [ 335982  335983  335984 ... 1426310 1426311 1426312]


#### (2) Create masks to filter unique IDs

In [6]:
total_runs_per_match = []

for mid in unique_matches:
    mask = (match_ids == mid)

    total_runs = np.sum(batsman_runs[mask])
    total_runs_per_match.append((mid, total_runs))

result = np.array(total_runs_per_match)
print("[(match_id, total_runs)]: ", result)

[(match_id, total_runs)]:  [[ 335982     268]
 [ 335983     430]
 [ 335984     244]
 ...
 [1426310     336]
 [1426311     301]
 [1426312     203]]


### Task 2: Top 5 Batters

#### (1) Get unique batters

In [7]:
unique_batters = np.unique(batters)

#### (2) Get runs per batter

In [8]:
total_runs_per_batter = [np.sum(batsman_runs[batters == batter]) for batter in unique_batters]
print("[(batter, total_runs)]: ", np.array(list(zip(unique_batters, total_runs_per_batter))))

[(batter, total_runs)]:  [['A Ashish Reddy' '280']
 ['A Badoni' '634']
 ['A Chandila' '4']
 ...
 ['Yudhvir Singh' '22']
 ['Yuvraj Singh' '2754']
 ['Z Khan' '117']]


#### (3) Sort total runs array

In [9]:
total_runs_array = np.array(total_runs_per_batter)
sorted_indices = np.argsort(total_runs_array)[::-1]
print("[Sorted(batter, total_runs)]: ", np.array(list(zip(unique_batters[sorted_indices], total_runs_array[sorted_indices]))))

[Sorted(batter, total_runs)]:  [['V Kohli' '8014']
 ['S Dhawan' '6769']
 ['RG Sharma' '6630']
 ...
 ['Yash Dayal' '0']
 ['U Kaul' '0']
 ['YA Abdulla' '0']]


#### (4) Get top 5 batters

In [10]:
top5_indices = sorted_indices[:5]
print("[Top 5 Batters]: ", np.array(list(zip(unique_batters[top5_indices], total_runs_array[top5_indices]))))

[Top 5 Batters]:  [['V Kohli' '8014']
 ['S Dhawan' '6769']
 ['RG Sharma' '6630']
 ['DA Warner' '6567']
 ['SK Raina' '5536']]


### Task 3: Strike Rate Calculation

#### (1) Compute Strike Rate

In [11]:
unique_batters, balls_faced = np.unique(batters, return_counts=True)
total_runs_array = np.array(total_runs_per_batter)
strike_rate = (total_runs_array / balls_faced) * 100
print("[(batter, strike_rate)]: ", np.array(list(zip(unique_batters, np.round(strike_rate, 2)))))

[(batter, strike_rate)]:  [['A Ashish Reddy' '142.86']
 ['A Badoni' '125.54']
 ['A Chandila' '57.14']
 ...
 ['Yudhvir Singh' '137.5']
 ['Yuvraj Singh' '124.78']
 ['Z Khan' '82.98']]


### Task 4: Economy Rate of Bowlers
#### (1) Calculate economy rate

In [12]:
unique_bowlers, balls_bowled = np.unique(bowlers, return_counts=True)
overs_bowled = balls_bowled / 6
total_runs_delivery = deliveries_data['total_runs']
runs_conceded = np.array([np.sum(total_runs_delivery[bowlers == b]) for b in unique_bowlers])
economy_rate = runs_conceded / overs_bowled
print("[(bowler, economy_rate)]: ", np.array(list(zip(unique_bowlers, np.round(economy_rate, 2)))))

[(bowler, economy_rate)]:  [['A Ashish Reddy' '8.89']
 ['A Badoni' '8.88']
 ['A Chandila' '6.28']
 ...
 ['Yudhvir Singh' '10.14']
 ['Yuvraj Singh' '7.42']
 ['Z Khan' '7.54']]


### Task 5: Runs per Over (1 to 20)
#### (1) Calculate average runs in each over

In [13]:
unique_overs = np.unique(overs)
avg_runs_per_over = []

for o in unique_overs:
    runs_in_over = batsman_runs[overs == o]
    matches_in_over = np.unique(match_ids[overs == o])
    avg_runs = np.sum(runs_in_over) / len(matches_in_over) if len(matches_in_over) > 0 else 0
    avg_runs_per_over.append(avg_runs)

print("[Avg Runs Per Over (1 to 20)]: ", np.array(list(zip(unique_overs + 1, np.round(avg_runs_per_over, 2)))))

[Avg Runs Per Over (1 to 20)]:  [[ 1.   11.32]
 [ 2.   13.6 ]
 [ 3.   15.46]
 [ 4.   15.99]
 [ 5.   16.21]
 [ 6.   16.13]
 [ 7.   12.83]
 [ 8.   13.95]
 [ 9.   14.56]
 [10.   14.31]
 [11.   14.77]
 [12.   15.07]
 [13.   15.1 ]
 [14.   15.58]
 [15.   16.01]
 [16.   16.25]
 [17.   16.77]
 [18.   17.28]
 [19.   17.  ]
 [20.   15.95]]


### Task 6: Boundary Analysis
#### (1) Count Fours and Sixes

In [14]:
fours_count = np.sum(batsman_runs == 4)
sixes_count = np.sum(batsman_runs == 6)
print("[Fours]: ", fours_count)
print("[Sixes]: ", sixes_count)

boundaries_mask = (batsman_runs == 4) | (batsman_runs == 6)
boundary_teams = batting_teams[boundaries_mask]
unique_teams, boundary_counts = np.unique(boundary_teams, return_counts=True)
max_boundary_team = unique_teams[np.argmax(boundary_counts)]
print("[Team with Most Boundaries]: ", max_boundary_team)

[Fours]:  29850
[Sixes]:  13051
[Team with Most Boundaries]:  Mumbai Indians


### Task 7: Death Overs Analysis (Overs 16-20)
#### (1) Analyze scoring in death overs

In [15]:
death_overs_mask = (overs >= 16) & (overs <= 20)
if not np.any(death_overs_mask):
    death_overs_mask = (overs >= 15) & (overs <= 19)

death_overs_runs = np.sum(batsman_runs[death_overs_mask])
print("[Total Runs in Death Overs]: ", death_overs_runs)

death_teams = batting_teams[death_overs_mask]
death_runs_array = batsman_runs[death_overs_mask]

unique_death_teams = np.unique(death_teams)
runs_per_death_team = [np.sum(death_runs_array[death_teams == team]) for team in unique_death_teams]
max_death_runs_team = unique_death_teams[np.argmax(runs_per_death_team)]
print("[Team with Highest Death Over Runs]: ", max_death_runs_team)

[Total Runs in Death Overs]:  71303
[Team with Highest Death Over Runs]:  Mumbai Indians


### Task 8: Highest Scoring Match
#### (1) Identify the highest scoring match

In [16]:
runs_column = result[:, 1].astype(int)
highest_match_idx = np.argmax(runs_column)
highest_scoring_match = result[highest_match_idx]
print("[Highest Scoring Match (match_id, total_runs)]: ", highest_scoring_match)

[Highest Scoring Match (match_id, total_runs)]:  [1426268     520]


### Task 9: Match Winner Approximation
#### (1) Determine winner based on runs

In [17]:
unique_match_ids = np.unique(match_ids)
approx_winners = []

for mid in unique_match_ids:
    match_mask = (match_ids == mid)
    teams_in_match = np.unique(batting_teams[match_mask])
    if len(teams_in_match) == 2:
        team1, team2 = teams_in_match
        runs_team1 = np.sum(batsman_runs[match_mask & (batting_teams == team1)])
        runs_team2 = np.sum(batsman_runs[match_mask & (batting_teams == team2)])
        
        if runs_team1 > runs_team2:
            winner = team1
        elif runs_team2 > runs_team1:
            winner = team2
        else:
            winner = "Tie"
        approx_winners.append((mid, winner))

print("[Match Winners (First 5)]: ", np.array(approx_winners))

[Match Winners (First 5)]:  [['335982' 'Kolkata Knight Riders']
 ['335983' 'Chennai Super Kings']
 ['335984' 'Tie']
 ...
 ['1426310' 'Tie']
 ['1426311' 'Sunrisers Hyderabad']
 ['1426312' 'Kolkata Knight Riders']]


### Task 10: Toss Impact Analysis
#### (1) Compare toss winners with match scoring

In [18]:
matches_data = np.genfromtxt(
    'dataset/matches.csv',
    delimiter=',',
    dtype=None,
    names=True,
    encoding='utf-8',
    invalid_raise=False,
    usecols=('id', 'toss_winner')
)

matches_id = matches_data['id']
toss_winners = matches_data['toss_winner']

toss_wins_more_runs = 0
total_matches_checked = 0

for mid, t_winner in zip(matches_id, toss_winners):
    match_mask = (match_ids == mid)
    teams_in_match = np.unique(batting_teams[match_mask])
    if len(teams_in_match) == 2:
        team1, team2 = teams_in_match
        runs_team1 = np.sum(batsman_runs[match_mask & (batting_teams == team1)])
        runs_team2 = np.sum(batsman_runs[match_mask & (batting_teams == team2)])
        
        approx_winner = team1 if runs_team1 > runs_team2 else (team2 if runs_team2 > runs_team1 else "Tie")
        if approx_winner == t_winner:
            toss_wins_more_runs += 1
        total_matches_checked += 1

print("[Matches where Toss Winner scored more runs]: ", toss_wins_more_runs)
print("[Total Matches Analyzed]: ", total_matches_checked)

[Matches where Toss Winner scored more runs]:  475
[Total Matches Analyzed]:  1092


### Task 11: Match Scorecard Generation
#### (1) Format and print scorecards

In [19]:
for mid in unique_match_ids:
    match_mask = (match_ids == mid)
    teams_in_match = np.unique(batting_teams[match_mask])
    
    print(f"Match {mid}:")
    for team in teams_in_match:
        team_runs = np.sum(batsman_runs[match_mask & (batting_teams == team)])
        print(f" {team}: {team_runs} runs")
    print()

Match 335982:
 Kolkata Knight Riders: 205 runs
 Royal Challengers Bangalore: 63 runs

Match 335983:
 Chennai Super Kings: 234 runs
 Kings XI Punjab: 196 runs

Match 335984:
 Delhi Daredevils: 122 runs
 Rajasthan Royals: 122 runs

Match 335985:
 Mumbai Indians: 154 runs
 Royal Challengers Bangalore: 161 runs

Match 335986:
 Deccan Chargers: 100 runs
 Kolkata Knight Riders: 84 runs

Match 335987:
 Kings XI Punjab: 162 runs
 Rajasthan Royals: 156 runs

Match 335988:
 Deccan Chargers: 137 runs
 Delhi Daredevils: 131 runs

Match 335989:
 Chennai Super Kings: 190 runs
 Mumbai Indians: 189 runs

Match 335990:
 Deccan Chargers: 208 runs
 Rajasthan Royals: 210 runs

Match 335991:
 Kings XI Punjab: 175 runs
 Mumbai Indians: 109 runs

Match 335992:
 Rajasthan Royals: 135 runs
 Royal Challengers Bangalore: 129 runs

Match 335993:
 Chennai Super Kings: 134 runs
 Kolkata Knight Riders: 139 runs

Match 335994:
 Deccan Chargers: 146 runs
 Mumbai Indians: 143 runs

Match 335995:
 Delhi Daredevils: 147 